# 05 — Carry and trend

Compare carry and trend with native systems, predeclared windows, costs, skew, drawdowns, and conservative exclusion evidence.

## Allocation and honest exclusions

This notebook replaces the sprawling carry-parameter and instrument-selection
project with one linear, inspectable experiment.  It asks four questions:

1. Is Chinese-futures trend P&L positively skewed in these data?
2. Are carry and trend actually weakly correlated, including bad months?
3. Does a larger carry forecast budget improve Sharpe without hiding worse
   tail risk or drawdown?
4. If we fit only through 2023-07-27, is any lineage or whole style so
   consistently loss-making that exclusion is defensible?

All 95 reviewed Chinese histories are present.  A new market enters as soon
as price, volatility, every core forecast, and the causal liquidity rule are
ready.  Present-day survivor labels never remove it.  The core is deliberately
small: four native absolute-carry smoothings (10/30/60/125) and three native
medium/slow EWMAC rules.  Earlier research found no robust reason to replace
that carry grid, and the 28-rule extension did not beat the carry-diversified
core convincingly enough to justify making it the production default.

## The rules are registered before looking at 2023–2026

- Fit window: 2008-07-28 through 2023-07-27.
- Revealed audit: 2023-07-28 through 2026-07-27.  It is already consumed and
  is **not** a fresh holdout.
- Carry-budget candidates are 0/20/40/50/60/70/80/100%.  Pure styles are
  diagnostics.  A 20–80% candidate replaces the 40% carry prior only if its
  pre-2023 robust annual-block score is at least 0.10 better; near-ties keep
  the candidate closest to 40%.
- A lineage/style is a “proven loser” only with at least eight complete
  July-to-July blocks, a loss in at least 75% of them, and a Bonferroni-adjusted
  one-sided 95% upper confidence bound for mean annual return below zero.
- Selection is only at lineage and whole-style level.  Choosing the best
  EWMAC or carry horizon separately for 95 instruments would be multiple-test
  overfitting, so every surviving style keeps all its horizons.

The systems use native forecasts, sizing, buffering, delayed whole-contract
fills, and stored cash/spread costs.  Only the dated portfolio gate is the
same thin repository helper introduced earlier in this series.  There is no
new `system.py`, YAML hierarchy, artifact framework, or hidden research helper.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import research as R

R.set_notebook_style()

In [ ]:
import gc
import math
from IPython.utils.io import capture_output
from scipy import stats

from sysdata.config.configdata import Config
from sysdata.sim.db_futures_sim_data import dbFuturesSimData
from sysobjects.adjusted_prices import futuresAdjustedPrices
from sysobjects.multiple_prices import futuresMultiplePrices
from sysobjects.spot_fx_prices import fxPrices
from systems.accounts.accounts_stage import Account
from systems.basesystem import System
from systems.forecast_combine import ForecastCombine
from systems.forecast_scale_cap import ForecastScaleCap
from systems.forecasting import Rules
from systems.positionsizing import PositionSizing
from systems.rawdata import RawData

R.limit_blas_threads()

FIT_START = pd.Timestamp("2008-07-28")
FIT_END = pd.Timestamp("2023-07-27")
AUDIT_START = pd.Timestamp("2023-07-28")
CUTOFF = pd.Timestamp("2026-07-27")
TRADING_DAYS = 256.0
CAPITAL = 100_000_000
TARGET_VOL = 16.0

## A visible database cutoff

The database may later contain more data.  This tiny subclass clips the three
time-series boundaries used by simulation and exposes only the reviewed
Tushare manifest.  It is intentionally here in the notebook rather than in a
project framework.

In [ ]:
class CutoffChinaData(dbFuturesSimData):
    def __init__(self, cutoff):
        self.cutoff = (
            pd.Timestamp(cutoff).normalize()
            + pd.Timedelta(days=1)
            - pd.Timedelta(nanoseconds=1)
        )
        manifest = R.TushareInstrumentManifest.from_csv()
        expected = sorted(
            item.instrument_code for item in manifest.mappings if item.is_stitchable
        )
        assert len(expected) == len(set(expected)) == 95
        self._instruments = tuple(expected)
        super().__init__()

        stored = set(super().get_instrument_list())
        missing = sorted(set(expected) - stored)
        if missing:
            raise ValueError(f"Database is missing reviewed instruments: {missing}")

        metadata = self.get_all_instrument_data_as_df().reindex(expected)
        bad = metadata.index[
            (metadata["Currency"] != "CNH") | (metadata["Region"] != "ASIA")
        ].tolist()
        if bad:
            raise ValueError(f"Non-Chinese instruments exposed: {bad}")

        listed = []
        for instrument in expected:
            raw = self.db_futures_adjusted_prices_data.get_adjusted_prices(
                instrument
            ).dropna()
            if len(raw) and raw.index[0] <= self.cutoff:
                listed.append(instrument)
        self._instruments = tuple(listed)

    def get_instrument_list(self):
        return list(self._instruments)

    def _check(self, instrument):
        if instrument not in self._instruments:
            raise ValueError(f"{instrument} is outside this cutoff universe")

    def get_backadjusted_futures_price(self, instrument_code):
        self._check(instrument_code)
        prices = super().get_backadjusted_futures_price(instrument_code)
        return futuresAdjustedPrices(pd.Series(prices.loc[: self.cutoff]).copy())

    def get_multiple_prices_from_start_date(self, instrument_code, start_date):
        self._check(instrument_code)
        prices = super().get_multiple_prices_from_start_date(
            instrument_code, start_date=start_date
        )
        return futuresMultiplePrices(pd.DataFrame(prices.loc[: self.cutoff]).copy())

    def _get_fx_data_from_start_date(self, currency1, currency2, start_date):
        prices = super()._get_fx_data_from_start_date(
            currency1, currency2, start_date=start_date
        )
        return fxPrices(pd.Series(prices.loc[: self.cutoff]).copy())


data = CutoffChinaData(CUTOFF)
ALL = R.chinese_universe(data)
assert len(ALL) == 95
assert max(data.daily_prices(code).index.max() for code in ALL) <= data.cutoff

# A predecessor and successor are one economic experiment, never two votes.
PREDECESSOR_SUCCESSOR = (
    ("CZCE_ME", "CZCE_MA"), ("CZCE_RO", "CZCE_OI"),
    ("CZCE_WT", "CZCE_PM"), ("CZCE_ER", "CZCE_RI"),
    ("CZCE_WS", "CZCE_WH"), ("CZCE_TC", "CZCE_ZC"),
    ("DCE_FB_OLD", "DCE_FB"),
)
LINEAGE = {instrument: instrument for instrument in ALL}
for predecessor, successor in PREDECESSOR_SUCCESSOR:
    LINEAGE[predecessor] = successor
assert len(set(LINEAGE.values())) == 88

print(f"{len(ALL)} Chinese instruments, {len(set(LINEAGE.values()))} lineages")
print(f"latest readable price date: {CUTOFF.date()}")

## Native rules and common controls

Fixed forecast scalars are the repository's long-run production fits.  They
normalise forecast amplitude; they do not optimise expected return.  The FDM
is fixed at one for every carry budget so the weight experiment is not quietly
helped by a separately fitted diversification multiplier.  Realised volatility
and a common-16%-volatility drawdown are both reported later.

The adjusted price is used in differences, never percentage changes.
Volatility backfilling is off, early warm-up stays missing, instrument weights
are equal across the currently eligible set, IDM is 2.5, capital is fixed at
100m CNH, and costs are the stored native cash/spread costs without future-vol
normalisation.

In [ ]:
EWMAC = "systems.provided.rules.ewmac.ewmac"
EWMAC_DATA = ["rawdata.get_daily_prices", "rawdata.daily_returns_volatility"]
CARRY = "systems.provided.rules.carry.carry"

TREND_RULES = ("ewmac16_64", "ewmac32_128", "ewmac64_256")
CARRY_RULES = ("carry10", "carry30", "carry60", "carry125")
ALL_RULES = TREND_RULES + CARRY_RULES

TRADING_RULES = {
    "ewmac16_64": dict(function=EWMAC, data=EWMAC_DATA,
                       other_args=dict(Lfast=16, Lslow=64)),
    "ewmac32_128": dict(function=EWMAC, data=EWMAC_DATA,
                        other_args=dict(Lfast=32, Lslow=128)),
    "ewmac64_256": dict(function=EWMAC, data=EWMAC_DATA,
                        other_args=dict(Lfast=64, Lslow=256)),
    "carry10": dict(function=CARRY, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=10)),
    "carry30": dict(function=CARRY, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=30)),
    "carry60": dict(function=CARRY, data=["rawdata.raw_carry"],
                    other_args=dict(smooth_days=60)),
    "carry125": dict(function=CARRY, data=["rawdata.raw_carry"],
                     other_args=dict(smooth_days=125)),
}
FORECAST_SCALARS = {
    "ewmac16_64": 3.75, "ewmac32_128": 2.65, "ewmac64_256": 1.87,
    "carry10": 27.82, "carry30": 28.38,
    "carry60": 28.40, "carry125": 29.37,
}


def weights_for(carry_budget):
    return {
        **{name: (1.0 - carry_budget) / len(TREND_RULES)
           for name in TREND_RULES},
        **{name: carry_budget / len(CARRY_RULES) for name in CARRY_RULES},
    }


def native_system(carry_budget, eligibility, fixed_weights,
                  forecast_weights_by_instrument=None):
    forecast_weights = (
        weights_for(carry_budget)
        if forecast_weights_by_instrument is None
        else forecast_weights_by_instrument
    )
    config = Config(dict(
        trading_rules=TRADING_RULES,
        forecast_scalars=FORECAST_SCALARS,
        forecast_weights=forecast_weights,
        use_forecast_scale_estimates=False,
        use_forecast_weight_estimates=False,
        forecast_div_multiplier=1.0,
        use_forecast_div_mult_estimates=False,
        instruments=ALL,
        instrument_weights={name: 1 / len(ALL) for name in ALL},
        instrument_div_multiplier=2.5,
        use_instrument_weight_estimates=False,
        use_instrument_div_mult_estimates=False,
        notional_trading_capital=CAPITAL,
        percentage_vol_target=TARGET_VOL,
        base_currency="CNH",
        capital_multiplier=dict(func="syscore.capital.fixed_capital"),
        buffer_method="forecast",
        buffer_size=0.10,
        buffer_trade_to_edge=True,
        forecast_cap=20.0,
        use_SR_costs=False,
        forecast_post_ceiling_cost_SR=999.0,
        vol_normalise_currency_costs=False,
        multiply_roll_costs_by=0.5,
        volatility_calculation=dict(
            func="sysquant.estimators.vol.mixed_vol_calc",
            name_returns_attr_in_rawdata="daily_returns",
            multiplier_to_get_daily_vol=1.0,
            days=35,
            min_periods=10,
            slow_vol_years=20,
            proportion_of_slow_vol=0.35,
            vol_abs_min=0.0000000001,
            backfill=False,
        ),
    ))
    return System(
        [Account(), R.PointInTimePortfolios(eligibility, fixed_weights),
         PositionSizing(), RawData(), ForecastCombine(),
         ForecastScaleCap(), Rules()],
        data,
        config,
    )


def accounting_frame(curve):
    frame = pd.concat({
        "gross": curve.percent.gross.as_ts,
        "costs": curve.percent.costs.as_ts,
        "net": curve.percent.as_ts,
    }, axis=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    error = (frame["net"] - frame["gross"] - frame["costs"]).abs().max()
    assert error < 1e-8
    return frame


def portfolio_frame(system):
    return accounting_frame(
        system.accounts.portfolio(delayfill=True, roundpositions=True)
    )

## One membership panel for every comparison

Liquidity enters at a 20-observed-session average of 130 contracts and exits
below 70.  Forecast readiness is kept separate because pandas can otherwise
turn an all-missing forecast panel into a plausible zero.  Every allocation
below trades exactly the same instruments on exactly the same dates.

In [ ]:
print("reading held-contract volume ...")
with capture_output():
    held_volume = R.held_contract_volumes(data, ALL)
liquidity = R.liquidity_eligibility(
    held_volume, force_terminal_close=True
)

print("checking all seven native forecasts for readiness ...")
probe = native_system(0.40, liquidity, R.equal_weight_panel(liquidity))
ready = {}
with capture_output():
    for instrument in ALL:
        forecasts = probe.combForecast.get_all_forecasts(instrument)
        subsystem = probe.portfolio.get_subsystem_position(instrument)
        aligned = pd.concat({
            "all forecasts": forecasts.notna().all(axis=1),
            "volatility and sizing": subsystem.notna(),
        }, axis=1).reindex(liquidity.index).ffill().fillna(False)
        ready[instrument] = aligned.all(axis=1)

ready = pd.DataFrame(ready, index=liquidity.index).astype(bool)
common_eligibility = liquidity & ready
common_weights = R.equal_weight_panel(common_eligibility)
first_investable = common_weights.index[
    common_weights.abs().sum(axis=1) > 0
][0]

assert not common_weights.where(~common_eligibility, 0.0).abs().to_numpy().any()
assert common_weights.loc[common_weights.sum(axis=1) > 0].sum(axis=1).sub(1).abs().max() < 1e-12

history = pd.DataFrame({
    "first price": {name: data.daily_prices(name).index[0] for name in ALL},
    "first eligible": {
        name: common_eligibility.index[common_eligibility[name]][0]
        if common_eligibility[name].any() else pd.NaT
        for name in ALL
    },
    "eligible days": common_eligibility.sum(),
})
print(f"{int((common_eligibility.sum() > 0).sum())} instruments ever trade; "
      f"{int(common_eligibility.iloc[-1].sum())} trade at the cutoff")
display(history.sort_values("first price").tail(12))

del probe
gc.collect()

## Carry-weight sweep

These are forecast budgets, not guaranteed realised risk shares.  Each system
is a complete native rerun, so forecast buffering, integer contracts, and costs
are allowed to respond to the different blend.  Pure carry and pure trend are
diagnostics; neither is eligible to become the production blend.  Full and
half-exposure CAGR mechanically compound the native fixed-capital daily P&L;
they are interpretable wealth illustrations, not separate variable-capital
contract reruns.

In [ ]:
CARRY_WEIGHTS = (0.0, 0.20, 0.40, 0.50, 0.60, 0.70, 0.80, 1.0)
allocation_frames = {}
style_systems = {}

for carry_weight in CARRY_WEIGHTS:
    print(f"running {carry_weight:.0%} carry / {1-carry_weight:.0%} trend ...")
    system = native_system(carry_weight, common_eligibility, common_weights)
    with capture_output():
        frame = portfolio_frame(system).loc[first_investable:CUTOFF]
    allocation_frames[carry_weight] = frame
    if carry_weight in (0.0, 1.0):
        style_systems[carry_weight] = system
    else:
        del system
        gc.collect()

print("allocation sweep complete")

In [ ]:
def sharpe(returns):
    clean = returns.dropna().astype(float)
    volatility = clean.std(ddof=1)
    if len(clean) < 2 or not np.isfinite(volatility) or volatility <= 0:
        return np.nan
    return clean.mean() / volatility * math.sqrt(TRADING_DAYS)


def compounded_stats(returns, exposure=1.0):
    clean = returns.dropna().astype(float) * exposure / 100.0
    if clean.empty:
        return np.nan, np.nan, np.nan
    wealth = (1.0 + clean).cumprod()
    anchored = pd.concat([
        pd.Series([1.0], index=[clean.index[0] - pd.Timedelta(nanoseconds=1)]),
        wealth,
    ])
    drawdown = anchored / anchored.cummax() - 1.0
    years = (clean.index[-1] - clean.index[0]).days / 365.25
    ending = float(wealth.iloc[-1])
    cagr = ending ** (1 / years) - 1 if years > 0 and ending > 0 else np.nan
    return 100 * (ending - 1), 100 * cagr, 100 * drawdown.min()


def performance(frame, start, end):
    sample = frame.loc[start:end].copy()
    net = sample["net"]
    annual_vol = net.std(ddof=1) * math.sqrt(TRADING_DAYS)
    cumulative = pd.concat([
        pd.Series([0.0], index=[net.index[0] - pd.Timedelta(nanoseconds=1)]),
        net.cumsum(),
    ])
    additive_drawdown = cumulative - cumulative.cummax()
    full_total, full_cagr, full_drawdown = compounded_stats(net, 1.0)
    half_total, half_cagr, half_drawdown = compounded_stats(net, 0.5)
    scaled = net * TARGET_VOL / annual_vol if annual_vol > 0 else net * np.nan
    _, _, common_vol_drawdown = compounded_stats(scaled, 1.0)
    return {
        "observations": len(sample),
        "net return %": net.sum(),
        "cost drag %": -sample["costs"].sum(),
        "net Sharpe": sharpe(net),
        "2x-cost Sharpe": sharpe(sample["gross"] + 2 * sample["costs"]),
        "ann vol %": annual_vol,
        "additive max DD %": additive_drawdown.min(),
        "full CAGR %": full_cagr,
        "full max DD %": full_drawdown,
        "half-exposure CAGR %": half_cagr,
        "half-exposure max DD %": half_drawdown,
        "16%-vol full max DD %": common_vol_drawdown,
    }


WINDOWS = {
    "post-2008": (FIT_START, CUTOFF),
    "fit through 2023-07-27": (FIT_START, FIT_END),
    "revealed last 3 years": (AUDIT_START, CUTOFF),
}
rows = []
for window, (start, end) in WINDOWS.items():
    for carry_weight, frame in allocation_frames.items():
        rows.append({
            "window": window,
            "carry weight": carry_weight,
            **performance(frame, start, end),
        })
allocation_summary = pd.DataFrame(rows).set_index(["window", "carry weight"])
display(allocation_summary[[
    "net Sharpe", "2x-cost Sharpe", "ann vol %", "additive max DD %",
    "full CAGR %", "full max DD %", "half-exposure CAGR %",
    "half-exposure max DD %", "16%-vol full max DD %",
]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for window in WINDOWS:
    table = allocation_summary.loc[window]
    axes[0].plot(100 * table.index, table["net Sharpe"], marker="o", label=window)
    axes[1].plot(100 * table.index, table["ann vol %"], marker="o", label=window)
    axes[2].plot(
        100 * table.index, table["16%-vol full max DD %"], marker="o", label=window
    )
axes[0].set_title("Net Sharpe")
axes[1].set_title("Resulting annual volatility")
axes[2].set_title("Full-compounding DD at common 16% vol")
for axis in axes:
    axis.set_xlabel("carry forecast budget (%)")
axes[0].legend()
plt.tight_layout()
plt.show()

## Choose the global carry budget using pre-2023 annual blocks

The score below is median annual-block Sharpe minus half its median absolute
deviation.  It rewards a repeatable result rather than one huge year.  The
40% carry prior stays unless another mixed candidate clears the predeclared
0.10 hurdle.  The latest three years have no vote.

In [ ]:
def annual_block_sharpes(frame):
    values = {}
    for year in range(2008, 2023):
        start = pd.Timestamp(year, 7, 28)
        end = pd.Timestamp(year + 1, 7, 27)
        sample = frame.loc[start:end, "net"]
        values[f"{year}-{year+1}"] = sharpe(sample)
    return pd.Series(values)


block_sharpes = pd.DataFrame({
    weight: annual_block_sharpes(frame)
    for weight, frame in allocation_frames.items()
})
score_rows = []
for weight in CARRY_WEIGHTS:
    values = block_sharpes[weight].dropna()
    median = values.median()
    mad = (values - median).abs().median()
    score_rows.append({
        "carry weight": weight,
        "median block Sharpe": median,
        "MAD": mad,
        "robust score": median - 0.5 * mad,
        "positive blocks": (values > 0).mean(),
    })
weight_scores = pd.DataFrame(score_rows).set_index("carry weight")

PRIOR = 0.40
selectable = weight_scores.loc[[0.20, 0.40, 0.50, 0.60, 0.70, 0.80]]
best_score = selectable["robust score"].max()
if best_score >= selectable.loc[PRIOR, "robust score"] + 0.10:
    near_best = selectable[
        selectable["robust score"] >= best_score - 0.10
    ].index
    CHOSEN_CARRY_WEIGHT = min(near_best, key=lambda value: abs(value - PRIOR))
    selection_reason = "a challenger cleared the 0.10 pre-2023 hurdle"
else:
    CHOSEN_CARRY_WEIGHT = PRIOR
    selection_reason = "no challenger cleared the 0.10 pre-2023 hurdle"

display(weight_scores)
block_sharpes.plot(kind="box", figsize=(11, 4.5),
                   title="July-to-July net Sharpe by carry forecast budget")
plt.xlabel("carry forecast budget")
plt.show()
print(f"Frozen global budget: {CHOSEN_CARRY_WEIGHT:.0%} carry / "
      f"{1-CHOSEN_CARRY_WEIGHT:.0%} trend because {selection_reason}.")

## Is trend positively skewed, and is carry really diversifying?

Skew depends on sampling horizon.  Daily skew alone is a poor description of
a slow trend trade, so the table reports daily, weekly, and monthly net P&L.
It also reports expected losses in the worst 5% of observations.  Correlation
is shown over the same horizons.  For monthly tails we report how often both
styles are simultaneously in their own bottom quintile (4% under independence)
and what one style makes during the other's bad months.  Low average
correlation is not protection if the two sleeves fail together in bad regimes.

In [ ]:
def aggregate(series, frequency):
    if frequency == "daily":
        return series
    rule = "W-FRI" if frequency == "weekly" else "M"
    return series.resample(rule).sum(min_count=1)


def expected_shortfall(series, probability=0.05):
    cutoff = series.quantile(probability)
    return series[series <= cutoff].mean()


diagnostic_rows = []
for window, (start, end) in WINDOWS.items():
    trend = allocation_frames[0.0].loc[start:end, "net"]
    carry = allocation_frames[1.0].loc[start:end, "net"]
    for frequency in ("daily", "weekly", "monthly"):
        pair = pd.concat({
            "carry": aggregate(carry, frequency),
            "trend": aggregate(trend, frequency),
        }, axis=1).fillna(0.0)
        periods_per_year = {
            "daily": TRADING_DAYS, "weekly": 52.0, "monthly": 12.0
        }[frequency]
        carry_period_vol = pair["carry"].std(ddof=1)
        trend_period_vol = pair["trend"].std(ddof=1)
        row = {
            "window": window,
            "frequency": frequency,
            "carry Sharpe": (
                pair["carry"].mean() / carry_period_vol
                * math.sqrt(periods_per_year)
            ),
            "trend Sharpe": (
                pair["trend"].mean() / trend_period_vol
                * math.sqrt(periods_per_year)
            ),
            "carry skew": pair["carry"].skew(),
            "trend skew": pair["trend"].skew(),
            "correlation": pair.corr().loc["carry", "trend"],
            "carry worst-5% mean": expected_shortfall(pair["carry"]),
            "trend worst-5% mean": expected_shortfall(pair["trend"]),
        }
        if frequency == "monthly":
            carry_bad = pair["carry"] <= pair["carry"].quantile(0.20)
            trend_bad = pair["trend"] <= pair["trend"].quantile(0.20)
            row["joint bottom-quintile frequency"] = (carry_bad & trend_bad).mean()
            row["carry mean when trend is bad"] = pair.loc[trend_bad, "carry"].mean()
            row["trend mean when carry is bad"] = pair.loc[carry_bad, "trend"].mean()
        diagnostic_rows.append(row)

style_diagnostics = pd.DataFrame(diagnostic_rows).set_index(
    ["window", "frequency"]
)
display(style_diagnostics)

In [ ]:
style_pair = pd.concat({
    "carry": allocation_frames[1.0]["net"],
    "trend": allocation_frames[0.0]["net"],
}, axis=1).fillna(0.0).loc[FIT_START:CUTOFF]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
R.cumulative_from_zero(style_pair).plot(
    ax=axes[0], title="Standalone carry and trend: cumulative fixed-capital P&L"
)
style_pair["carry"].rolling(256, min_periods=128).corr(
    style_pair["trend"]
).plot(ax=axes[1], title="Rolling one-year daily P&L correlation")
axes[1].axhline(0.0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

monthly_pair = style_pair.resample("M").sum(min_count=1)
monthly_pair.plot.scatter(
    x="trend", y="carry", title="Monthly trend versus carry net P&L"
)
plt.axhline(0.0, color="black", linewidth=0.7)
plt.axvline(0.0, color="black", linewidth=0.7)
plt.show()

## Which instruments made money from carry and trend recently?

The table is descriptive and includes all 95 names.  “Insufficient” means
fewer than 100 common-eligibility sessions in the three-year window, not a
loss.  A recent positive Sharpe is never fed back into the exclusion mask.

In [ ]:
def instrument_frames(system):
    answer = {}
    with capture_output():
        for instrument in ALL:
            curve = system.accounts.pandl_for_instrument(
                instrument, delayfill=True, roundpositions=True
            )
            answer[instrument] = accounting_frame(curve)
    return answer


print("extracting native instrument P&L for standalone carry and trend ...")
trend_instruments = instrument_frames(style_systems[0.0])
carry_instruments = instrument_frames(style_systems[1.0])


def style_label(carry_sharpe, trend_sharpe, sessions):
    if sessions < 100 or not np.isfinite(carry_sharpe + trend_sharpe):
        return "insufficient"
    if carry_sharpe > 0 and trend_sharpe > 0:
        return "both"
    if carry_sharpe > 0:
        return "carry only"
    if trend_sharpe > 0:
        return "trend only"
    return "neither"


instrument_rows = []
for instrument in ALL:
    sessions = int(common_eligibility.loc[AUDIT_START:CUTOFF, instrument].sum())
    carry_recent = carry_instruments[instrument].loc[AUDIT_START:CUTOFF, "net"]
    trend_recent = trend_instruments[instrument].loc[AUDIT_START:CUTOFF, "net"]
    carry_sr = sharpe(carry_recent) if sessions >= 100 else np.nan
    trend_sr = sharpe(trend_recent) if sessions >= 100 else np.nan
    instrument_rows.append({
        "instrument": instrument,
        "lineage": LINEAGE[instrument],
        "recent eligible sessions": sessions,
        "recent carry return %": carry_recent.sum(),
        "recent carry Sharpe": carry_sr,
        "recent trend return %": trend_recent.sum(),
        "recent trend Sharpe": trend_sr,
        "recent result": style_label(carry_sr, trend_sr, sessions),
    })

instrument_style = pd.DataFrame(instrument_rows).set_index("instrument")
print(instrument_style["recent result"].value_counts().to_string())
display(instrument_style.sort_values(["recent result", "recent carry Sharpe"],
                                     ascending=[True, False]))

In [ ]:
heatmap = instrument_style[["recent carry Sharpe", "recent trend Sharpe"]].copy()
heatmap = heatmap.sort_values("recent carry Sharpe")
fig, ax = plt.subplots(figsize=(8, 20))
image = ax.imshow(heatmap.to_numpy(), aspect="auto", cmap="RdYlGn",
                  vmin=-2, vmax=2)
ax.set_yticks(range(len(heatmap)))
ax.set_yticklabels(heatmap.index, fontsize=7)
ax.set_xticks([0, 1])
ax.set_xticklabels(["carry", "trend"])
ax.set_title("Revealed 2023–2026 net Sharpe by instrument and style")
fig.colorbar(image, ax=ax, label="net Sharpe", shrink=0.4)
plt.tight_layout()
plt.show()

## “Proven negative” is deliberately hard to satisfy

For each lineage and for carry, trend, and the chosen mixed core, we sum native
instrument P&L inside completed July-to-July blocks.  A block counts only with
at least 150 eligible sessions.  The confidence test is on annual net return,
which has the same sign as expected Sharpe when volatility is positive.

Bonferroni uses all 88 × 3 possible lineage/style tests, not merely the rows
that happened to look bad.  This is conservative by design.  If it finds no
proven losers, the correct output is “exclude none.”

In [ ]:
print(f"running the chosen {CHOSEN_CARRY_WEIGHT:.0%}-carry core for attribution ...")
core_system = native_system(
    CHOSEN_CARRY_WEIGHT, common_eligibility, common_weights
)
core_frame = portfolio_frame(core_system).loc[first_investable:CUTOFF]
core_instruments = instrument_frames(core_system)


def lineage_series(instrument_data):
    result = {}
    for lineage in sorted(set(LINEAGE.values())):
        members = [name for name in ALL if LINEAGE[name] == lineage]
        result[lineage] = pd.concat(
            [instrument_data[name]["net"] for name in members], axis=1
        ).sum(axis=1, min_count=1).fillna(0.0)
    return result


def lineage_allowed(lineage):
    members = [name for name in ALL if LINEAGE[name] == lineage]
    return common_eligibility[members].any(axis=1)


model_lineages = {
    "carry": lineage_series(carry_instruments),
    "trend": lineage_series(trend_instruments),
    "core": lineage_series(core_instruments),
}

NUMBER_OF_TESTS = len(set(LINEAGE.values())) * len(model_lineages)
ALPHA_PER_TEST = 0.05 / NUMBER_OF_TESTS
proof_rows = []
for model, lineages in model_lineages.items():
    for lineage, returns in lineages.items():
        allowed = lineage_allowed(lineage)
        annual_returns = []
        daily_pieces = []
        for year in range(2008, 2023):
            start = pd.Timestamp(year, 7, 28)
            end = pd.Timestamp(year + 1, 7, 27)
            block_allowed = allowed.loc[start:end]
            if int(block_allowed.sum()) < 150:
                continue
            block = returns.loc[start:end]
            annual_returns.append(float(block.sum()))
            daily_pieces.append(block)

        n_blocks = len(annual_returns)
        mean_return = np.mean(annual_returns) if n_blocks else np.nan
        loss_fraction = (
            np.mean(np.asarray(annual_returns) < 0) if n_blocks else np.nan
        )
        if n_blocks >= 2:
            standard_error = np.std(annual_returns, ddof=1) / math.sqrt(n_blocks)
            critical = stats.t.ppf(1 - ALPHA_PER_TEST, n_blocks - 1)
            upper_bound = mean_return + critical * standard_error
        else:
            upper_bound = np.nan
        pooled = pd.concat(daily_pieces) if daily_pieces else pd.Series(dtype=float)
        proven = bool(
            n_blocks >= 8 and loss_fraction >= 0.75
            and np.isfinite(upper_bound) and upper_bound < 0
        )
        proof_rows.append({
            "model": model,
            "lineage": lineage,
            "complete blocks": n_blocks,
            "pooled net Sharpe": sharpe(pooled),
            "mean annual contribution %": mean_return,
            "losing block fraction": loss_fraction,
            "Bonferroni upper 95% bound": upper_bound,
            "proven negative": proven,
        })

proof = pd.DataFrame(proof_rows).set_index(["model", "lineage"]).sort_index()
assert (proof.loc[proof["proven negative"], "complete blocks"] >= 8).all()
assert (proof.loc[proof["proven negative"], "losing block fraction"] >= 0.75).all()
assert (proof.loc[proof["proven negative"], "Bonferroni upper 95% bound"] < 0).all()

proven_table = proof[proof["proven negative"]]
print(f"Bonferroni alpha per test: {ALPHA_PER_TEST:.6g}")
print(f"proven-negative lineage/style rows: {len(proven_table)}")
display(proven_table if len(proven_table) else
        proof.sort_values("Bonferroni upper 95% bound").head(15))

## Freeze the simple mask, then launch the revealed audit flat

A proven-negative core removes the lineage.  Otherwise a proven-negative
carry or trend sleeve removes that entire style for the lineage and reallocates
the forecast budget to the surviving style.  If both styles look proven
negative but the combined core does not, we keep the core: contradictory
selection evidence is not permission to delete the market.

Lineages with fewer than eight blocks—including new listings—must retain the
global rule mix.  The audit systems hold zero positions before 2023-07-28,
delay the first trade, and pay its native entry cost.

In [ ]:
EXCLUDED_LINEAGES = set(
    proof.xs("core").index[proof.xs("core")["proven negative"]]
)
REMOVE_CARRY = set(
    proof.xs("carry").index[proof.xs("carry")["proven negative"]]
) - EXCLUDED_LINEAGES
REMOVE_TREND = set(
    proof.xs("trend").index[proof.xs("trend")["proven negative"]]
) - EXCLUDED_LINEAGES

contradictory = REMOVE_CARRY & REMOVE_TREND
REMOVE_CARRY -= contradictory
REMOVE_TREND -= contradictory

core_blocks = proof.xs("core")["complete blocks"]
short_history = set(core_blocks.index[core_blocks < 8])
assert not short_history & (EXCLUDED_LINEAGES | REMOVE_CARRY | REMOVE_TREND)

weights_by_instrument = {}
for instrument in ALL:
    lineage = LINEAGE[instrument]
    if lineage in REMOVE_CARRY:
        weights_by_instrument[instrument] = weights_for(0.0)
    elif lineage in REMOVE_TREND:
        weights_by_instrument[instrument] = weights_for(1.0)
    else:
        weights_by_instrument[instrument] = weights_for(CHOSEN_CARRY_WEIGHT)

print("excluded lineages:", sorted(EXCLUDED_LINEAGES) or "none")
print("carry removed:", sorted(REMOVE_CARRY) or "none")
print("trend removed:", sorted(REMOVE_TREND) or "none")
print(f"{len(short_history)} short-history lineages keep the universal mix")

audit_allowed = common_eligibility.copy()
audit_allowed.loc[audit_allowed.index < AUDIT_START] = False
selected_allowed = audit_allowed.copy()
for instrument in ALL:
    if LINEAGE[instrument] in EXCLUDED_LINEAGES:
        selected_allowed[instrument] = False

baseline_weights = R.equal_weight_panel(audit_allowed)
selected_weights = R.equal_weight_panel(selected_allowed)

baseline_audit_system = native_system(
    CHOSEN_CARRY_WEIGHT, audit_allowed, baseline_weights
)
selected_audit_system = native_system(
    CHOSEN_CARRY_WEIGHT,
    selected_allowed,
    selected_weights,
    forecast_weights_by_instrument=weights_by_instrument,
)

print("running the two flat-launch audit systems ...")
with capture_output():
    baseline_audit = portfolio_frame(baseline_audit_system).loc[AUDIT_START:CUTOFF]
    selected_audit = portfolio_frame(selected_audit_system).loc[AUDIT_START:CUTOFF]

In [ ]:
# Prove the launch convention on actual native positions and costs.
entry_checks = []
for instrument in ALL:
    target = baseline_audit_system.accounts.get_buffered_position(
        instrument, roundpositions=True
    ).fillna(0.0)
    curve = baseline_audit_system.accounts.pandl_for_instrument(
        instrument, delayfill=True, roundpositions=True
    )
    calculator = curve.pandl_calculator_with_costs
    actual = calculator.positions.fillna(0.0)
    assert actual.loc[actual.index < AUDIT_START].eq(0.0).all()

    targets = target.index[(target.index >= AUDIT_START) & target.ne(0.0)]
    fills = sorted(
        [fill for fill in calculator.fills
         if pd.Timestamp(fill.date) >= AUDIT_START and abs(fill.qty) > 0],
        key=lambda fill: fill.date,
    )
    if len(targets) and len(fills):
        costs = calculator.costs_from_trading_in_instrument_currency_as_series()
        fill_date = pd.Timestamp(fills[0].date)
        fill_cost = float(costs.loc[fill_date])
        entry_checks.append((instrument, pd.Timestamp(targets[0]),
                             fill_date, fill_cost))

assert entry_checks
first_entry = sorted(entry_checks, key=lambda row: row[2])[0]
assert first_entry[2] > first_entry[1]
assert first_entry[3] < 0
print("first target, delayed fill, native cost:", first_entry)

In [ ]:
audit_summary = pd.DataFrame({
    "universal": performance(baseline_audit, AUDIT_START, CUTOFF),
    "proven-negative mask": performance(selected_audit, AUDIT_START, CUTOFF),
}).T
display(audit_summary[[
    "net return %", "cost drag %", "net Sharpe", "2x-cost Sharpe",
    "ann vol %", "additive max DD %", "full CAGR %", "full max DD %",
    "half-exposure CAGR %", "half-exposure max DD %",
]])

R.cumulative_from_zero(pd.DataFrame({
    "universal": baseline_audit["net"],
    "proven-negative mask": selected_audit["net"],
})).plot(title="Revealed audit: flat-launch cumulative net P&L")
plt.axhline(0.0, color="black", linewidth=0.7)
plt.show()

In [ ]:
selected_contributions = []
with capture_output():
    for instrument in ALL:
        frame = accounting_frame(
            selected_audit_system.accounts.pandl_for_instrument(
                instrument, delayfill=True, roundpositions=True
            )
        ).loc[AUDIT_START:CUTOFF]
        selected_contributions.append({
            "instrument": instrument,
            "lineage": LINEAGE[instrument],
            "net contribution %": frame["net"].sum(),
            "gross contribution %": frame["gross"].sum(),
            "cost drag %": -frame["costs"].sum(),
            "net Sharpe": sharpe(frame["net"]),
        })

selected_contributions = pd.DataFrame(selected_contributions).set_index("instrument")
assert abs(
    selected_contributions["net contribution %"].sum()
    - selected_audit["net"].sum()
) < 1e-8
display(pd.concat([
    selected_contributions.nsmallest(15, "net contribution %"),
    selected_contributions.nlargest(15, "net contribution %"),
]))

## Decision

The next cell writes the conclusion from the calculated tables so the prose
cannot silently outlive refreshed data.  Remember that low carry/trend
correlation is a diversification benefit, not proof that carry deserves an
unbounded weight.  Carry is exposed to crowded yield harvesting, curve and
roll measurement error, policy interventions, limit moves, seasonality, and
joint liquidation shocks; those dangers show up more clearly in skew,
expected shortfall, joint-tail frequency, and drawdown than in Sharpe alone.

In [ ]:
post2008_daily = style_diagnostics.loc[("post-2008", "daily")]
post2008_weekly = style_diagnostics.loc[("post-2008", "weekly")]
post2008_monthly = style_diagnostics.loc[("post-2008", "monthly")]
recent_carry = allocation_summary.loc[("revealed last 3 years", 1.0), "net Sharpe"]
recent_trend = allocation_summary.loc[("revealed last 3 years", 0.0), "net Sharpe"]

print(
    f"Trend skew was {post2008_daily['trend skew']:+.2f} daily, "
    f"{post2008_weekly['trend skew']:+.2f} weekly, and "
    f"{post2008_monthly['trend skew']:+.2f} monthly."
)
print(
    f"Carry/trend correlation was {post2008_daily['correlation']:+.2f} daily, "
    f"{post2008_weekly['correlation']:+.2f} weekly, and "
    f"{post2008_monthly['correlation']:+.2f} monthly. Both styles were in "
    f"their own bottom quintile in "
    f"{post2008_monthly['joint bottom-quintile frequency']:.1%} of months "
    "(4% is the independence benchmark)."
)
print(
    f"Recent standalone Sharpe: carry {recent_carry:.2f}, trend {recent_trend:.2f}. "
    f"The pre-2023 rule froze {CHOSEN_CARRY_WEIGHT:.0%} carry."
)
print(
    f"The proof rule excluded {len(EXCLUDED_LINEAGES)} lineages, removed carry "
    f"from {len(REMOVE_CARRY)}, and removed trend from {len(REMOVE_TREND)}."
)
print(
    f"Flat-launch recent Sharpe was {audit_summary.loc['universal', 'net Sharpe']:.2f} "
    f"universally and {audit_summary.loc['proven-negative mask', 'net Sharpe']:.2f} "
    "after the frozen proof rule."
)
print()
print(
    "Recommendation: keep the transparent carry+EWMAC core as the production "
    "baseline, use the pre-2023 global carry budget above, include new markets "
    "by default, and make proven-negative exclusions rare. Keep the 28-rule "
    "system as a shadow diversifier until independent forward evidence pays "
    "for its extra model and operational risk."
)

## Durable lessons from the earlier carry and selection studies

Carry results are sensitive to smoothing, costs, and sample boundaries; a
single full-sample winner is not a rule-selection argument. Likewise, deleting
markets because their completed histories lost money is hindsight. A durable
exclusion must be predeclared, causal, and strong enough to survive the
multiple comparisons across lineages and styles. The familywise test above is
the canonical version of those conclusions.